# FraudFinder — Iteration 1 Data Exploration
**The Okies | ACS 5513 | Fall 2026**

This notebook verifies the uploaded Kaggle credit card transaction dataset, describes data quality, and documents a proposed preparation strategy. No model is trained here.

## 1. Load the dataset
Put `creditcard.csv` in the same folder as this notebook, or update `DATA_PATH`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
DATA_PATH = Path('creditcard.csv')
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())

## 2. Dataset structure and quality

In [ ]:
print('Data types:')
print(df.dtypes.value_counts())
print('Missing cells:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
print('Unique values in Class:', sorted(df['Class'].unique().tolist()))
display(df[['Time','Amount']].describe())

## 3. Fraud class distribution
The data are extremely imbalanced. A model predicting every transaction as legitimate would score about 99.83% accuracy but detect no fraud.

In [ ]:
counts = df['Class'].value_counts().sort_index()
display(pd.DataFrame({'count': counts, 'percent': counts / len(df) * 100}))
ax = counts.plot(kind='bar', title='Transactions by class (log scale)', logy=True)
ax.set_xticklabels(['Legitimate (0)', 'Fraud (1)'], rotation=0)
ax.set_ylabel('Transaction count')
plt.tight_layout()
plt.show()

## 4. Compare transaction amounts
Note: the units/currency are not specified in the public data description. We should not assume dollars.

In [ ]:
display(df.groupby('Class')['Amount'].agg(['count','mean','median','min','max']))
ax = df.boxplot(column='Amount', by='Class', showfliers=False)
ax.set_title('Amount by class (outliers hidden)')
ax.set_xlabel('Class')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 5. Duplicate investigation
Exact duplicates can reflect duplicate records, but this anonymized dataset does not give us enough information to know whether all are errors. We will examine their class breakdown before choosing a removal policy.

In [ ]:
dupes = df.duplicated(keep='first')
print('Duplicate rows beyond first occurrence:', int(dupes.sum()))
display(df.loc[dupes, 'Class'].value_counts().rename('duplicate_rows'))

## 6. Preparation plan (not model results)
- Keep the test set untouched during training and model selection.
- Use a stratified train/test split to preserve rare fraud cases.
- Decide whether and how to remove duplicates **before** splitting, to avoid exact copies appearing in both sets.
- Fit any scaling on training data only, ideally using a scikit-learn Pipeline.
- Evaluate precision, recall, F1, PR-AUC, and the confusion matrix; do not rely on accuracy alone.
- Record decisions, changes, and evidence in the team report.

In [ ]:
from sklearn.model_selection import train_test_split
# Example preparation split, without training a model:
X = df.drop(columns='Class')
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Train:', X_train.shape, 'Fraud:', int(y_train.sum()))
print('Test:', X_test.shape, 'Fraud:', int(y_test.sum()))
# IMPORTANT: revisit duplicate handling before finalizing the split.

## AI use and validation
ChatGPT helped structure the notebook and explain the analyses. The team should run every cell, verify counts against the CSV, review the plots, and document any changes before submission.